[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kewel-corp/book-vol2/blob/main/notebooks/ch20_code.ipynb)

# 第20章　眼科①：眼底写真と糖尿病網膜症

**『医療診断支援AI開発　第2巻　応用編 ― 本格実装から臨床導入まで（第2巻　応用編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-vol2

## 20.3　マルチラベルという設計上の要点

```text
Root_Dataset/
├── train/
│   ├── images/                # 元画像（.jpg）
│   └── masks/
│       ├── MA/                # 微小血管瘤の二値マスク（.tif）
│       ├── Hemo/              # 出血
│       ├── HardExudate/       # 硬性白斑
│       └── SoftExudate/       # 軟性白斑
├── valid/                     # trainと同構成
└── test/
```

## 20.5　実装の構成

In [ ]:
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp

# U-Netベースのマルチラベルセグメンテーションモデル
model = smp.Unet(
    encoder_name="efficientnet-b3",  # EfficientNetをエンコーダに
    encoder_weights="imagenet",       # 事前学習の重みを利用
    in_channels=3,                    # RGB入力
    classes=4,                        # MA, Hemo, Hard, Soft の4病変
    activation=None,                  # 出力は後でSigmoid（マルチラベル）
)

# Dice損失（マルチラベル）＋ BCE。DiceLoss はバッチ全体で陽性がないチャネルを外すので、
# 陰性だけのバッチでも偽陽性を抑えられるよう、ロジットに直接かける BCE を足す
dice_loss = smp.losses.DiceLoss(mode="multilabel", from_logits=True)
bce_loss = nn.BCEWithLogitsLoss()

def loss_fn(logits, target):          # target は logits と同じ形の 0/1（float）。0 は「その病変なし」と確認済みの画素に限る（未注釈を0で埋めない）
    return dice_loss(logits, target) + bce_loss(logits, target)

# 病変ごとに独立したマスクを積み重ねて (4, H, W) の教師とする
def build_target(masks_dict):
    channels = [masks_dict[k] for k in ["MA", "Hemo", "Hard", "Soft"]]
    return torch.stack(channels, dim=0).float()

## スクリーニングの判定基準：referable DR と国際重症度分類

In [ ]:
# 病変マップ→紹介判定の集約（説明可能なルート）と、直接グレーディング（高感度ルート）の併用
def referable_decision(lesion_stats, grade_logits, refer_thr=0.35,
                       heavy_hemo=20, moderate_or_worse=(2, 3, 4)):
    # lesion_stats: {"MA":count, "Hemo":count, "Hard":.., "Soft":.., "Hard_at_macula":..} 病変ごとの検出数/面積
    # moderate_or_worse: ICDR 0〜4 のうち中等症NPDR以上に当たるクラスindex
    # 注意：これは4-2-1ルールそのものではない。4-2-1は象限別の条件なので、
    # 実装するには象限ごとの病変数が要る。ここは「多発出血による簡易な紹介トリガー」。
    hemorrhage_heavy = lesion_stats["Hemo"] >= heavy_hemo    # 眼底全体での出血数
    macular_exudate  = lesion_stats["Hard_at_macula"] > 0    # 黄斑近傍の硬性白斑＝DME疑い
    # softmax はクラス軸だけで取り、sum もクラス軸だけで取る。全体を sum すると
    # バッチの複数症例を足し合わせてしまう（この関数は1症例ぶんの入力を前提とする）。
    grade_prob = grade_logits.softmax(-1)[..., moderate_or_worse].sum(-1)
    refer = (grade_prob >= refer_thr) or hemorrhage_heavy or macular_exudate
    return {"refer": bool(refer), "grade_prob": float(grade_prob),
            "reason": {"heavy_hemorrhage": hemorrhage_heavy, "dme_suspect": macular_exudate}}

## 追加ケース：緑内障スクリーニングと視神経乳頭の定量

In [ ]:
import numpy as np
from scipy import ndimage

def vertical_cdr(disc_mask, cup_mask):
    """同じ画像座標の2D二値マスク（0/1またはbool）を受け取り、計測値と状態を返す。
    本例では空マスク・分断・陥凹の乳頭外へのはみ出しを評価不能とする。
    空の陥凹予測だけから、生理的なCDR=0とは判定しない。
    """
    disc, cup = np.asarray(disc_mask), np.asarray(cup_mask)
    def invalid(reason):
        return {"vertical_cdr": None, "state": "評価不能（" + reason + "）"}
    if disc.ndim != 2 or cup.ndim != 2 or disc.shape != cup.shape:
        return invalid("マスクの次元または形状が不一致")
    if not (np.isin(disc, [0, 1]).all() and np.isin(cup, [0, 1]).all()):
        return invalid("二値マスクでない")
    disc, cup = disc.astype(bool), cup.astype(bool)
    if not disc.any() or not cup.any():
        return invalid("乳頭または陥凹のマスクが空")
    connected = np.ones((3, 3), dtype=int)      # 本例では8近傍を採用
    if any(ndimage.label(m, structure=connected)[1] != 1 for m in (disc, cup)):
        return invalid("マスクが複数の成分に分断")
    if np.any(cup & ~disc):
        return invalid("陥凹が乳頭の外にはみ出す")
    def v_extent(m):
        rows = np.flatnonzero(m.any(axis=1))
        return int(rows[-1] - rows[0] + 1)     # 上端から下端までの範囲長
    return {"vertical_cdr": v_extent(cup) / v_extent(disc), "state": "ok"}
# CDR に加え、リム(disc-cup)の厚みが ISNT則（下≧上≧鼻≧耳）を満たすかも見る